# IR Lab Tutorial: Statistical Analysis

This tutorial shows how to conduct a hypothesis test to compare two retrieval approaches.
The two runs compared in this example are loaded from the TIRA cache.

## Step 1: Ensure that libraries are imported

In [33]:
# This command loads and starts PyTerrier so that it also works in TIRA.

from tira.third_party_integrations import ensure_pyterrier_is_loaded

ensure_pyterrier_is_loaded()

In [34]:
# PyTerrier must be imported after `ensure_pyterrier_is_loaded` is called.

from pyterrier import started, init

if not started():
    init()

## Step 2: Load the dataset

In [35]:
from pyterrier import get_dataset

dataset_train = get_dataset('irds:ir-lab-wise-2024/subsampled-ms-marco-deep-learning-20241201-training')
dataset_train

IRDSDataset('ir-lab-wise-2024/subsampled-ms-marco-deep-learning-20241201-training')

In [36]:
dataset_validation = get_dataset('irds:ir-lab-wise-2024/subsampled-ms-marco-rag-20250105-training')
dataset_validation

IRDSDataset('ir-lab-wise-2024/subsampled-ms-marco-rag-20250105-training')

In [37]:
dataset_test = get_dataset('irds:ir-lab-wise-2024/subsampled-ms-marco-ir-lab-20250105-test')
dataset_test

IRDSDataset('ir-lab-wise-2024/subsampled-ms-marco-ir-lab-20250105-test')

In [38]:
# For development, let's use the training set for the experiments.
dataset = dataset_train

## Step 3: Create the retrieval pipeline with TIRA

In this example, we will use two existing submitted runs and load the approaches via the TIRA API.

In [39]:
from tira.rest_api_client import Client

tira_client = Client()

The approach IDs below follow the structure: `<task>/<team>/<submission>`

In [47]:
approach_baseline = tira_client.pt.from_retriever_submission(
    approach='ir-lab-wise-2024/ir-wise-24-guugel/Hyperparameter baseline',
    dataset='subsampled-ms-marco-deep-learning-20241201-training',
)
approach_baseline

TiraSourceTransformer()

In [48]:
approach_baseline = tira_client.pt.from_retriever_submission(
    approach='ir-lab-wise-2024/ir-wise-24-tutors/Retrieval Baseline',
    dataset='subsampled-ms-marco-deep-learning-20241201-training',
)
approach_baseline

Download: 1.11MiB [00:00, 2.74MiB/s]


Download finished. Extract...
Extraction finished:  /root/.tira/extracted_runs/ir-lab-wise-2024/subsampled-ms-marco-deep-learning-20241201-training/ir-wise-24-tutors


TiraSourceTransformer()

In [49]:
approach_new = tira_client.pt.from_retriever_submission(
    approach='ir-lab-wise-2024/ir-wise-24-guugel/Hyperparameter tuning',
    dataset='subsampled-ms-marco-deep-learning-20241201-training',
)
approach_new

TiraSourceTransformer()

In [56]:
approach_new = tira_client.pt.from_retriever_submission(
    approach='ir-lab-wise-2024/ir-wise-24-th25/BM25 + MonoT5 Rerank',
    dataset='subsampled-ms-marco-deep-learning-20241201-training',
)
approach_new

Download: 1.37MiB [00:00, 3.34MiB/s]


Download finished. Extract...
Extraction finished:  /root/.tira/extracted_runs/ir-lab-wise-2024/subsampled-ms-marco-deep-learning-20241201-training/ir-wise-24-th25


TiraSourceTransformer()

## Step 4: Measure effectiveness

Now let us measure the nDCG@10 effectiveness of both systems on the Touché 2020 task 1 dataset.

In [57]:
from pyterrier.pipelines import Experiment

experiment = Experiment(
    retr_systems=[
        approach_baseline,
        approach_new,
    ],
    topics=dataset_train.get_topics("query"),
    qrels=dataset_train.get_qrels(),
    eval_metrics=["ndcg_cut_10"],
    names=[
        "Baseline Approach",
        "New Approach",
    ],
    perquery=True,
)
experiment.sample(n=10)

,name,qid,measure,value
165,New Approach,1117099,ndcg_cut_10,1.000000
168,New Approach,359349,ndcg_cut_10,0.801154
51,Baseline Approach,938400,ndcg_cut_10,0.098864
170,New Approach,1112341,ndcg_cut_10,0.641751
179,New Approach,490595,ndcg_cut_10,0.274095
1,Baseline Approach,1037496,ndcg_cut_10,0.544501
151,New Approach,156493,ndcg_cut_10,0.939215
126,New Approach,174463,ndcg_cut_10,0.709677
45,Baseline Approach,701453,ndcg_cut_10,0.911156
111,New Approach,1121353,ndcg_cut_10,0.846997


This data frame shows the nDCG@10 values measured for each query and both systems. \
So we have pairs of measurements where the same metric (i.e., nDCG@10) is measured using the same input (e.g., query #1) but for two different systems.
Let's re-arrange the data frame so that the effectiveness values are in separate columns, not rows.

In [58]:
experiment_baseline = experiment[experiment["name"] == "Baseline Approach"]\
    .drop(columns=["name"])
experiment_approach = experiment[experiment["name"] == "New Approach"]\
    .drop(columns=["name"])

experiment_paired = experiment_baseline.merge(
    experiment_approach,
    on=["qid", "measure"],
    suffixes=("_baseline", "_approach"),
)
experiment_paired.head(n=10)

,qid,measure,value_baseline,value_approach
0,1030303,ndcg_cut_10,0.872746,0.627356
1,1037496,ndcg_cut_10,0.544501,0.912539
2,1037798,ndcg_cut_10,0.152866,0.182572
3,1043135,ndcg_cut_10,0.607376,0.756584
4,104861,ndcg_cut_10,0.777951,1.000000
5,1051399,ndcg_cut_10,0.084364,0.821843
6,1063750,ndcg_cut_10,0.268899,0.841178
7,1064670,ndcg_cut_10,0.824434,0.757463
8,1071750,ndcg_cut_10,0.417623,0.705113
9,1103812,ndcg_cut_10,0.593333,0.738388


## Step 5: Conduct hypothesis tests

On this _paired_ measurement data, we can now conduct _paired_ t-tests to test for statistical significance of given hypotheses.
Remember that the choice of your test depends (amongst other factors) on how the hypothesis is formulated.

Let us test some hypotheses to get a feeling of what this means:

#### Hypothesis 1: The new approach has a significantly different nDCG@10 on the chosen dataset than the baseline.
(Hint: For your own tests, you'd want to replace the approach and dataset names with the actual names above.)

Significance test: two-sided paired t-test \
Significance level: $\alpha = 0.05$ (i.e., the effect is only considered significant if $p < 0.05$)

In [59]:
from scipy.stats import ttest_rel

ttest_rel(
    experiment_paired["value_approach"],
    experiment_paired["value_baseline"],
    alternative='two-sided',
).pvalue

5.3937220780435224e-14

The above value is called $p$, the probability of the corresponding null hypothesis (the probability that the effect would be observed by chance). \
If this is lower than our significance level $\alpha$, we can reject the null hypothesis and confirm the hypothesis 1.

Now it would be great to find out which is better. \
One way could be to formulate a hypothesis with a predefined "direction". In this example we assume our new approach to be better.

#### Hypothesis 2: The new approach has a significantly higher nDCG@10 on the chosen dataset than the baseline.

Significance test: one-sided paired t-test \
Significance level: $\alpha = 0.05$ (or $p < 0.05$)

In [60]:
from scipy.stats import ttest_rel

ttest_rel(
    experiment_paired["value_approach"],
    experiment_paired["value_baseline"],
    alternative='greater',
).pvalue

2.6968610390217612e-14

Again, if the probability $p$ of the null hypothesis is lower than our significance level $\alpha$, then we can reject the null hypothesis and confirm hypothesis 2.

Let us test the opposite direction: the new approach could be worse w.r.t. nDCG@10 than the baseline.

#### Hypothesis 2: The new approach has a significantly lower nDCG@10 on the chosen dataset than the baseline.

Significance test: one-sided paired t-test \
Significance level: $\alpha = 0.05$ (or $p < 0.05$)

In [61]:
from scipy.stats import ttest_rel

ttest_rel(
    experiment_paired["value_approach"],
    experiment_paired["value_baseline"],
    alternative='less',
).pvalue

0.999999999999973

Again, if the probability $p$ of the null hypothesis is lower than our significance level $\alpha$, then we can reject the null hypothesis and confirm hypothesis 3.